# RadCluster_2_1 — Digital-Twin Campaign Control (T3 rev 6)

**Run this notebook unchanged on every participating machine. Nothing in it
is machine-specific and nothing needs editing.**

The host identifies itself from `machines.json`, which is also the single
source of the frozen grid — the notebook *builds its command line from that
file* rather than carrying its own copy, so the grid cannot drift between
machines and is never retyped. `git pull` in section 1 is the only way
settings change.

Run sections in order. **0–4 are pre-flight and must all pass before 5
launches the real run.**


## 0 — Setup


In [1]:
import json, subprocess, sys, time, collections
from pathlib import Path

def _find_root():
    """Locate digital_twin from wherever the kernel happens to start."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base, base / 'RadCluster_2_1' / 'digital_twin'):
            if (cand / 'run_ensemble.py').exists() and (cand / 'machines.json').exists():
                return cand
    raise SystemExit('cannot locate RadCluster_2_1/digital_twin from ' + str(Path.cwd()))

HERE = _find_root()
sys.path.insert(0, str(HERE))
import campaign_ops as ops, run_ensemble as RE

REPO     = Path(subprocess.run(['git','rev-parse','--show-toplevel'], cwd=HERE,
                               capture_output=True, text=True).stdout.strip())
REGISTRY = HERE / 'machines.json'
RESULTS  = HERE / 'results'
PY       = sys.executable
print('repo :', REPO)
print('here :', HERE)


repo : D:\GitHub\RadCluster
here : d:\GitHub\RadCluster\RadCluster_2_1\digital_twin


## 1 — Pull

Every participant must run the same code, design and grid. `merge_and_sobol`
compares `git_sha`, `solver_sha256`, `workbook_sha256`, `design_sha256`,
`run_cfg_sha`, `weights_sha` and `of`, and reports a PROVENANCE SPLIT on any
disagreement. **Re-run this cell whenever settings change — that is the only
step needed to pick up a new grid.**


In [2]:
print(subprocess.run(['git','pull','--ff-only'], cwd=REPO,
                     capture_output=True, text=True).stdout.strip())
print('HEAD =', subprocess.run(['git','rev-parse','--short','HEAD'], cwd=REPO,
                               capture_output=True, text=True).stdout.strip())


Already up to date.
HEAD = 739a160


## 2 — Read the frozen campaign settings

Everything below is derived from `machines.json`. Nothing is hard-coded here.


In [ ]:
reg    = json.loads(REGISTRY.read_text())
G      = reg['grid']
DESIGN = HERE / reg['design']
EXPECTED_SHA = reg['run_cfg_sha']

GRID = ['--equations', G['equations'],
        '--I', str(G['I']), '--V', str(G['V']),
        '--i-discrete', str(G['i_discrete']), '--v-discrete', str(G['v_discrete']),
        '--i-bin', str(G['i_bin']), '--v-bin', str(G['v_bin']),
        '--shape-function', G['shape_function'],
        '--i-mobile-default', str(G['i_mobile_default']),
        '--v-mobile-default', str(G['v_mobile_default']),
        '--dose', str(G['dose']), '--rtol', str(G['rtol']),
        '--solver-mode', G['solver_mode']]

# The ROW BUDGET is deliberately NOT in GRID any more.  It used to be passed
# as --timeout-s reg['timeout_s'] -- the GLOBAL value -- from this notebook,
# which silently defeated the per-machine `timeout_s` that entries 1 and 3
# had carried since 2026-08-07: the flag always won, so the override only
# ever applied when somebody retyped it on a hand-built command line.
# run_ensemble now resolves it itself, --timeout-s > machine entry > global,
# and PRINTS the value it settled on.  Leave the flag off here so the
# registry stays the single source of truth (§2's promise).
_mine = [m for m in reg['machines']
         if m['index'] == RE.detect_machine(reg, RE._host_facts())['index']]
TIMEOUT_S = (_mine[0].get('timeout_s') if _mine else None) or reg['timeout_s']

print('design      ', DESIGN.name)
print('grid        ', ' '.join(f'{k}={v}' for k, v in G.items()))
print('run_cfg_sha ', EXPECTED_SHA)
print('timeout_s   ', TIMEOUT_S,
      '(this machine)' if _mine and _mine[0].get('timeout_s') else '(global default)')
print('\nNOTE: i_mobile is PHYSICS, not a numerical knob. These rows cannot be')
print('pooled with any run at a different i_mobile; run_cfg_sha enforces it.')
print('NOTE: timeout_s is a RESOURCE budget, excluded from run_cfg_sha, so a')
print('per-machine value does not split provenance or repartition the design.')


## 3 — Who am I?

Detection is by host fingerprint and **fails loudly** on no match or an
ambiguous match. A wrong index means two machines compute the same rows and
some rows are computed by nobody — which surfaces at merge time as *rows
MISSING*, indistinguishable from a machine that never reported.

If it refuses: `RE.register_host(<index>, REGISTRY)`, then commit and push
`machines.json`. Do not guess an index.


In [4]:
me    = RE.detect_machine(reg, RE._host_facts())   # raises if unrecognised
W     = [float(w) for w in reg['weights'].split(',')]
rows, meta = RE.read_design(DESIGN)
mine  = [r for r in rows if RE.assign_machine(int(r['row_id']), reg['of'], W) == me['index']]
print(f"machine {me['index']} = {me['name']}   slots {me['slots']}   "
      f"speed {me['speed']} ({me['speed_source']})   weight {me['weight']}")
print(f'owns {len(mine)} of {len(rows)} rows')


machine 2 = Nasr Workstation   slots 14   speed 0.5 (DECLARED)   weight 7.0
owns 240 of 1008 rows


## 4 — Build, agreement gate, and a bounded PRE-FLIGHT

**Do not skip the pre-flight.** Its absence cost two aborted Hoffman2
submissions: every row failed in 5–8 s with `d100=nan` while the provenance
line looked perfect — that line prints *before* any row runs and proves only
that the config was assembled.

The cause is unresolved and **may affect any Linux host**: rows succeed
through a direct call but fail through `run_ensemble`'s multiprocessing pool
at the production grid. So it must run *on this machine*.


In [5]:
info = ops.ensure_solver()
r = subprocess.run([PY, str(HERE / 'check_machine.py')], cwd=HERE,
                   capture_output=True, text=True)
print(r.stdout[-1800:])
assert r.returncode == 0, 'AGREEMENT GATE FAILED — do not contribute rows from this build'


  solver: OK  D:\GitHub\RadCluster\RadCluster_2_1\build\Release\solver.exe  sha c28893e7d4a119e2


machine   Nasr-Workstation  (Windows-11-10.0.26200-SP0)
python    3.14.3
git       739a160c4ab0
solver    c28893e7d4a119e2  exists=True
workbook  9253e7a0370af966

running probe (I=150, 0.02 dpa, ~30 s) ...

  reference generated on Nasr-Workstation (git 76efc2a46f6c, solver c28893e7d4a119e2)
  note: git SHA differs from the reference machine - pull first if that is not intentional.

  field                    this machine        reference    rel diff
  Di_eff                7.085348836e-12  7.085348836e-12    0.00e+00
  Dv_eff                2.125201773e-13  2.125201773e-13    0.00e+00
  conv_psuccess         2.181172484e-06  2.181172484e-06    0.00e+00
  conv_psuccess_abs     1.000000000e+00  1.000000000e+00    0.00e+00
  N_loops_100           1.679711678e+20  1.679711678e+20    0.00e+00
  N_loops_111           2.904994914e+23  2.904994914e+23    0.00e+00
  mean_n_100            2.016423189e+02  2.016423189e+02    0.00e+00
  mean_n_111            3.604016025e+01  3.604016025e+01    0

In [ ]:
import re as _re
t0 = time.time()
pf = RESULTS / '_preflight.jsonl'
# No --timeout-s: the preflight must exercise the SAME budget the production
# run will use, which is now resolved from machines.json inside run_ensemble.
# --workers 2 is honoured again as of 2026-08-08; it used to be silently
# overwritten by the registry's `slots`, so this "2-row, 2-worker" gate was in
# fact running with the machine's full slot count.
r  = subprocess.run([PY, '-u', str(HERE / 'run_ensemble.py'),
                     '--design', str(DESIGN), '--machine', 'auto', *GRID,
                     '--limit', '2', '--workers', '2', '--out', str(pf)],
                    cwd=HERE, capture_output=True, text=True)
out = r.stdout + r.stderr
print(out[-1800:])
m   = _re.search(r'"run_cfg_sha": "([0-9a-f]+)"', out)
sha = m.group(1) if m else None
print(f'\n  run_cfg_sha {sha}  (expected {EXPECTED_SHA})')
print(f'  FAIL lines  {out.count("FAIL")}')
print(f'  elapsed     {time.time()-t0:.0f} s   -> per-row cost sets the ETA')
assert sha == EXPECTED_SHA, 'GRID MISMATCH — this machine is not on the frozen grid'
assert out.count('FAIL') == 0, 'ROWS FAILED — do NOT launch; report the error text'
for p in RESULTS.glob('_preflight*'): p.unlink()
print('\n  PRE-FLIGHT PASSED')


## 5 — Launch

Detached, so the notebook can be closed. Resumption is the design: re-running
this cell skips `row_id`s already present and only ever adds.

> The row budget is a budget, not a cap. On expiry the solver is asked to
> finalize and its **partial trajectory is kept**, contributing to every rung
> of the dose ladder it reached.
>
> It is no longer passed from here. `run_ensemble` resolves it
> `--timeout-s` → this machine's `timeout_s` in `machines.json` → the
> top-level `timeout_s` → 3600 s, and prints what it chose. That is what makes
> a per-machine budget possible: it is a *resource* setting, excluded from
> `run_cfg_sha`, so it neither splits provenance nor repartitions the design.
>
> `simulation.py` has enforced it as a **row-level** deadline since
> 2026-08-06, and on this host it bites cleanly (rows cut at 12 062 s under a
> 12 000 s budget). **It does not on machine 0**, which reports full-dose rows
> of up to 27 254 s under a 3600 s budget — an 8× overrun that is unexplained
> and means machine 0's walls cannot be read as budget-limited.


In [ ]:
ops.clear_stop()
LOG = RESULTS / f"worker_machine{me['index']}.log"
cmd = [PY, '-u', str(HERE / 'run_ensemble.py'), '--design', str(DESIGN),
       '--machine', 'auto', *GRID]
print(' '.join(cmd), '\n')
with open(LOG, 'w') as fh:
    proc = subprocess.Popen(cmd, cwd=HERE, stdout=fh, stderr=subprocess.STDOUT,
                            start_new_session=True)
print(f'launched pid {proc.pid} -> {LOG.name}')
# Read the log, do not just trust the pid.  A worker that dies in main() --
# as every Windows host did between 2026-08-07 and 2026-08-08, on a bare
# signal.SIGHUP that does not exist off POSIX -- prints the provenance line
# first, so "it launched" and "it is running" look identical for one screen.
# The row budget line below the provenance line is the proof it got past it.
time.sleep(40)
_log = open(LOG).read()
print(_log[:1200])
assert 'Traceback' not in _log, 'WORKER DIED AT STARTUP - read the log above, do not walk away'


## 6 — Monitor


In [ ]:
ops.watch(DESIGN, RESULTS, n_machines=reg['of'], interval=120)   # Ctrl-C to stop watching


  refreshed 05:05:34  (every 120s; interrupt the kernel to stop watching)

CAMPAIGN  T3_rev6.csv   p=19 N=16 conditions=N2,N5,I1
  progress  [####################..........................]  43.9%  443/1008 rows
            admissible 443   inadmissible 0   failed 0   missing 565

  timing    per row  mean 2h28m   median 2h33m   p90 3h57m
            core-hours used 1097.3   remaining ~1399.4

   machine                 id  assigned   done   left     ETA @14w
         0  MacBook-Pro.local       480    228    252       44h35m
         1                  -        55      0     55        9h43m
         2   Nasr-Workstation       240    112    128       22h38m
         3     Mac.san.rr.com       233    103    130       22h59m

  *** PROVENANCE SPLIT on git_sha - results are NOT comparable:
        1d652999fae45a3e  <- ['MacBook-Pro.local']
        f704409db6cc94f3  <- ['MacBook-Pro.local']
        303ee6121900a50f  <- ['Nasr-Workstation']
        739a160c4ab0e757  <- ['Mac.san.rr.com']
   

In [ ]:
st = ops.campaign_status(DESIGN, RESULTS, n_machines=reg['of'])
ops.render_status(st, ops.load_targets())


### 6b — Throughput and dose-ladder coverage

Throughput, not wall-time-per-row, is the measure that survives concurrency —
planning from per-row wall measured at low concurrency is what produced a
3780 s estimate against a 17 828 s reality.


In [ ]:
recs = ops.load_results(RESULTS)
walls = [r['wall_s'] for r in recs.values() if r.get('wall_s') and not r.get('solver_rc')]
if walls:
    import statistics
    mean = statistics.mean(walls)
    cap  = sum(float(w) for w in reg['weights'].split(','))
    print(f'rows done {len(walls)}   mean wall {mean:.0f} s   '
          f"throughput {me['slots']*3600/mean:.2f} rows/h on this machine")
    print(f'projected campaign: {len(rows)*mean/cap/3600:.0f} h over {cap:.1f} slot-equivalents')
cov = collections.Counter()
for r in recs.values():
    for rung in (r.get('at_dose') or {}): cov[rung] += 1
for k in sorted(cov, key=float): print(f'  {k:>4s} dpa : {cov[k]:4d} rows')


## 7 — Graceful stop / resume

Rows in flight finish and are written; no new rows start.


In [ ]:
ops.request_stop('put the real reason here')


In [ ]:
ops.clear_stop()


## 8 — Pool and report (one machine, after everyone has pushed)

Push `results/*.jsonl` **and** `*.manifest.json` — the manifest is what sizes
the next campaign from measured throughput.

Run it **both ways**. If the parameter *ranking* is unchanged with and without
`--require-converged`, truncation did not buy it anything — the empirical form
of the claim the no-gating policy rests on.


In [ ]:
for extra in ([], ['--require-converged']):
    print('='*70); print('  merge_and_sobol', *extra); print('='*70)
    r = subprocess.run([PY, str(HERE / 'merge_and_sobol.py'),
                        '--design', str(DESIGN), '--results', str(RESULTS),
                        '--at-dose', str(reg['grid']['dose']), *extra],
                       cwd=HERE, capture_output=True, text=True)
    print(r.stdout[-4000:])
